# ZTF SNe Ia — DESI Galaxy Cross-Match via LS_ID

This notebook links ZTF Type Ia supernovae to DESI spectroscopic galaxy data using the Legacy Survey ID (`LS_ID`) as the matching key.

**Input files:**
- `data/ztf_mosthost_match_filtered_lc_ls_id.csv` — ZTF SNe matched to MOST Host catalog (contains `mosthost_ls_id_dr9`)
- `desi_bright_galaxy_metadata_mosthost.fits` — DESI bright survey galaxy metadata (contains `LS_ID`)
- `desi_dark_galaxy_metadata_mosthost.fits` — DESI dark survey galaxy metadata (contains `LS_ID`)

**Strategy:**
1. Load the ZTF–MOST Host matched CSV
2. Load both DESI FITS catalogs and combine them
3. Match on `LS_ID`, prefixing all DESI columns with `DESI_`
4. Inspect and save the result

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
from astropy.io import fits
from astropy.table import Table, vstack

## 2. Load the ZTF–MOST Host matched catalog

In [2]:
ztf = pd.read_csv("data/ztf_mosthost_match_filtered_lc_ls_id.csv")
ztf.head()

,ztf_ztfname,ztf_redshift,ztf_redshift_err,ztf_source,ztf_t0,ztf_x0,ztf_x1,ztf_c,ztf_t0_err,ztf_x0_err,...,mosthost_ba_leda_sga,mosthost_z_leda_sga,mosthost_ref_sga,mosthost_group_id_sga,mosthost_group_name_sga,mosthost_group_ra_sga,mosthost_group_dec_sga,mosthost_group_diameter_sga,mosthost_d26_sga,mosthost_d26_ref_sga
0,ZTF18aaadqua,0.078672,0.002807,z_snid,58130.778798,0.000845,4.999999,-0.392734,14.743977,0.000103,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ZTF18aaadqua,0.078672,0.002807,z_snid,58130.778798,0.000845,4.999999,-0.392734,14.743977,0.000103,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ZTF18aaadqua,0.078672,0.002807,z_snid,58130.778798,0.000845,4.999999,-0.392734,14.743977,0.000103,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ZTF18aabasml,0.047269,0.000013,z_gal,58171.942251,0.000458,4.999980,1.008026,0.805199,0.000099,...,0.651628,0.04721,LEDA-20181114,368300.0,PGC033565,166.48647,37.625363,0.428549,0.548347,SB26
4,ZTF18aabasml,0.047269,0.000013,z_gal,58171.942251,0.000458,4.999980,1.008026,0.805199,0.000099,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 2a. Cast `mosthost_ls_id_dr9` to integer

The CSV stores LS_IDs in scientific notation (float), but the FITS files store them as integers. We need to cast to `int64` for a clean merge. Rows with NaN or zero LS_ID cannot be matched, so we flag them.

In [3]:
# Cast to int64 for matching
ztf["mosthost_ls_id_dr9"] = ztf["mosthost_ls_id_dr9"].fillna(0).astype(np.int64)

## 3. Load both DESI FITS catalogs

We read the bright and dark survey metadata, then stack them into a single DataFrame. Rows with `LS_ID == 0` have no Legacy Survey counterpart and are dropped before matching.

In [4]:
desi_dir = "../../myc21_first_paper_LARGE_DATA"

# Read bright survey
with fits.open("../../myc21_first_paper_LARGE_DATA/desi_bright_galaxy_metadata_mosthost.fits") as hdul:
    bright = Table(hdul[1].data).to_pandas()
    print(f"Bright survey rows: {len(bright)}")

# Read dark survey
with fits.open("../../myc21_first_paper_LARGE_DATA/desi_dark_galaxy_metadata_mosthost.fits") as hdul:
    dark = Table(hdul[1].data).to_pandas()
    print(f"Dark survey rows: {len(dark)}")

Bright survey rows: 6346725
Dark survey rows: 8490171


In [5]:
# Combine bright + dark
desi = pd.concat([bright, dark], ignore_index=True)
print(f"Combined DESI rows: {len(desi)}")

# Drop rows with no LS_ID (cannot match)
desi = desi[desi["LS_ID"] != 0].copy()
print(f"After dropping LS_ID == 0: {len(desi)}")

# Check for duplicates on LS_ID
n_dup = desi["LS_ID"].duplicated().sum()
print(f"Duplicate LS_IDs in DESI: {n_dup}")

desi.head()

Combined DESI rows: 14836896
After dropping LS_ID == 0: 14814759
Duplicate LS_IDs in DESI: 184697


,TARGETID,LS_ID,SPECTYPE,RA,DEC,DESI_TARGET,Z,ZWARN,SURVEY
7,2376326874923008,9906621356250415,GALAXY,127.453557,-0.205956,4611686018427387904,0.246758,4,main
8,2376357954715648,9906621841870006,GALAXY,179.910462,0.957400,4611686018427387904,1.023395,0,main
13,2376855818600448,9906629620994209,GALAXY,6.985185,22.324928,4611686018427387904,1.671872,4,main
14,2376933706825728,9906630837997223,GALAXY,37.047157,25.627123,4611686018427387904,0.494737,4,main
15,2376953432637440,9906631146214958,GALAXY,263.804724,26.578494,4611686018427387904,1.027075,4,main


## 4. Prefix DESI columns and merge

We rename all DESI columns with a `DESI_` prefix (except the merge key `LS_ID`) to avoid collisions, then perform a left join on `LS_ID`.

In [6]:
# Rename DESI columns with prefix
desi_renamed = desi.rename(
    columns={col: f"DESI_{col}" for col in desi.columns if col != "LS_ID"}
)

print(f"DESI columns after renaming: {list(desi_renamed.columns)}")

DESI columns after renaming: ['DESI_TARGETID', 'LS_ID', 'DESI_SPECTYPE', 'DESI_RA', 'DESI_DEC', 'DESI_DESI_TARGET', 'DESI_Z', 'DESI_ZWARN', 'DESI_SURVEY']


In [7]:
# Merge: ZTF (left) <-> DESI (right) on LS_ID
merged = ztf.merge(
    desi_renamed,
    left_on="mosthost_ls_id_dr9",
    right_on="LS_ID",
    how="left",
    indicator=True
)

print(merged["_merge"].value_counts())

_merge
left_only     1121
both           132
right_only       0
Name: count, dtype: int64


## 5. Inspect the results

In [8]:
matched = merged[merged["_merge"] == "both"].copy()
unmatched = merged[merged["_merge"] == "left_only"].copy()

print(f"Total ZTF rows:      {len(ztf)}")
print(f"Matched to DESI:     {len(matched)}")
print(f"Unmatched:           {len(unmatched)}")
print(f"Unique matched SNe:  {matched['ztf_ztfname'].nunique()}")

matched.head()

Total ZTF rows:      1251
Matched to DESI:     132
Unmatched:           1121
Unique matched SNe:  126


,ztf_ztfname,ztf_redshift,ztf_redshift_err,ztf_source,ztf_t0,ztf_x0,ztf_x1,ztf_c,ztf_t0_err,ztf_x0_err,...,DESI_TARGETID,LS_ID,DESI_SPECTYPE,DESI_RA,DESI_DEC,DESI_DESI_TARGET,DESI_Z,DESI_ZWARN,DESI_SURVEY,_merge
69,ZTF18aamlqqh,0.057183,0.000021,z_gal,58242.685183,0.001290,-1.372104,-0.057532,0.091402,0.000041,...,3.963298e+16,9.907733e+15,GALAXY,266.986007,34.932719,5.764608e+18,0.057182,0.0,main,both
72,ZTF18aaobguk,0.130312,0.000015,z_gal,58239.371581,0.000275,-0.073190,-0.126726,0.117252,0.000009,...,3.963327e+16,9.907738e+15,GALAXY,178.049495,51.365305,5.764608e+18,0.237021,0.0,main,both
81,ZTF18aapqwyv,0.056010,0.000022,z_gal,58244.195740,0.000696,-1.842507,0.096945,0.088507,0.000022,...,3.962839e+16,9.906631e+15,GALAXY,251.605420,25.421488,1.152922e+18,0.056008,0.0,main,both
91,ZTF18aaqcugm,0.061863,0.000016,z_gal,58254.947400,0.001171,-0.954839,-0.092637,0.049551,0.000035,...,3.963331e+16,9.907739e+15,GALAXY,202.053416,53.911700,5.764608e+18,0.061870,0.0,main,both
110,ZTF18aaqskoy,0.086084,0.000011,z_gal,58255.068169,0.000828,1.067525,-0.111924,0.075822,0.000025,...,3.963301e+16,9.907734e+15,GALAXY,255.907053,36.428186,2.594000e+03,0.086084,0.0,main,both


## 6. Clean up and save

In [9]:
# Drop the merge indicator and redundant LS_ID column
matched = matched.drop(columns=["_merge"])

output_path = "data/ztf_desi_crossmatched.csv"
matched.to_csv(output_path, index=False)
print(f"Saved {len(matched)} rows to {output_path}")
print(f"Final columns: {list(matched.columns)}")

Saved 132 rows to data/ztf_desi_crossmatched.csv
Final columns: ['ztf_ztfname', 'ztf_redshift', 'ztf_redshift_err', 'ztf_source', 'ztf_t0', 'ztf_x0', 'ztf_x1', 'ztf_c', 'ztf_t0_err', 'ztf_x0_err', 'ztf_x1_err', 'ztf_c_err', 'ztf_cov_t0_x0', 'ztf_cov_t0_x1', 'ztf_cov_t0_c', 'ztf_cov_x0_x1', 'ztf_cov_x0_c', 'ztf_cov_x1_c', 'ztf_mwebv', 'ztf_mwr_v', 'ztf_mwebv_err', 'ztf_fitprob', 'ztf_ra', 'ztf_dec', 'ztf_sn_type', 'ztf_sub_type', 'ztf_lccoverage_flag', 'ztf_fitquality_flag', 'ztf_iau_name', 'ztf_frac_fitted', 'mosthost_sn_name_sp', 'mosthost_hostnum', 'mosthost_ra', 'mosthost_dec', 'mosthost_origin', 'mosthost_sn_type', 'mosthost_sn_z', 'mosthost_sn_ra', 'mosthost_sn_dec', 'mosthost_sn_name', 'mosthost_sn_name_ptf', 'mosthost_sn_name_iau', 'mosthost_sn_name_tns', 'mosthost_program', 'mosthost_ls_id_dr9', 'mosthost_dec_dr9', 'mosthost_ra_dr9', 'mosthost_ref_id_dr9', 'mosthost_brickid_dr9', 'mosthost_ref_cat_dr9', 'mosthost_type_dr9', 'mosthost_dist_arcsec_dr9', 'mosthost_sep_in_radius_dr